# KernC on a real GPU (Google Colab)

Runtime → **Change runtime type → T4 GPU**, then run the cells in order.

This builds the KernC compiler, then runs the whole verification suite, the fuzzer and mutation testing **with real GPU execution**: every compiled kernel runs on the T4 and its output must match the AST reference interpreter bit-for-bit.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,driver_version --format=csv
!which ptxas || ls /usr/local/cuda/bin/ptxas

## 1. Upload `kernc.zip`
(Or replace this cell with `!git clone https://github.com/<you>/kernc.git` once you have pushed it.)

In [ ]:
from google.colab import files
import os, zipfile
up = files.upload()          # choose kernc.zip
name = next(iter(up))
zipfile.ZipFile(name).extractall('/content')
os.chdir('/content/kernc')
print(os.getcwd(), os.listdir('.'))

## 2. Build

In [ ]:
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build -j4 2>&1 | tail -3
!./build/bin/kernc --version
!./build/bin/kernc_unit_tests | tail -1

## 3. Compile a kernel and look at the PTX

In [ ]:
!./build/bin/kernc compile examples/saxpy.kern -O1

## 4. Run one kernel on the GPU and on the CPU oracle

In [ ]:
!python3 tools/gpu_run.py tests/exec/saxpy_golden.kern
!python3 tools/make_launch.py tests/exec/saxpy_golden.kern > /tmp/saxpy.launch
!./build/bin/kernc run tests/exec/saxpy_golden.kern --launch /tmp/saxpy.launch --engine ast

In [ ]:
!python3 tools/demo.py --width 100 --height 36

## 5. Full verification suite on the GPU

In [ ]:
!python3 tools/run_tests.py --gpu

## 6. Differential fuzzing on the GPU

In [ ]:
!python3 tools/fuzz.py -n 300 --gpu

## 7. Mutation testing (with GPU execution)
The second command runs the fuzzer alone. On the CPU it cannot see code-generation bugs; on the GPU it can now kill mutants such as #13 (wrong index scaling) and usually #12 (NaN `!=`). Mutant #14 (missing `cvta.to.global`) will most likely still survive: on current GPUs generic and global addresses coincide, so that bug is invisible at run time and only the static FileCheck test catches it.

In [ ]:
!python3 tools/mutate.py --gpu --fuzz 100
!python3 tools/mutate.py --gpu --no-tests --fuzz 150 --report /tmp/fuzz_only.md

## 8. Optimiser benchmark

In [ ]:
!python3 tools/bench.py